# 🍏 Observability & Tracing Demo with `azure-ai-projects` and `azure-ai-inference` 🍎

Welcome to this **Health & Fitness**-themed notebook, where we'll explore how to set up **observability** and **tracing** for:

1. **Basic LLM calls** using an `AIProjectClient`.
2. **Multi-step** interactions using an **Agent** (such as a Health Resource Agent).
3. **Tracing** your local usage in **console** (stdout) or via an **OTLP endpoint** (like **Prompty** or **Aspire**).
4. Sending those **traces** to **Azure Monitor** (Application Insights) so you can view them in **Azure AI Foundry**.

> **Disclaimer**: This is a fun demonstration of AI and observability! Any references to workouts, diets, or health routines in the code or prompts are purely for **educational** purposes. Always consult a professional for health advice.

## Contents
1. **Initialization**: Setting up environment, creating clients.
2. **Basic LLM Call**: Quick demonstration of retrieving model completions.
3. **Connections**: Listing project connections.
4. **Observability & Tracing**
   - **Console / Local** tracing
   - **Prompty / Aspire**: piping traces to a local OTLP endpoint
   - **Azure Monitor** tracing: hooking up to Application Insights
   - **Verifying** your traces in Azure AI Foundry
5. **Agent-based Example**:
   - Creating a simple "Health Resource Agent" referencing sample docs.
   - Multi-turn conversation with tracing.
   - Cleanup.

<img src="./seq-diagrams/1-observability.png" width="50%"/>

## 1. Initialization & Setup
**Prerequisites**:
- A `.env` file containing `PROJECT_ENDPOINT` (and optionally `MODEL_DEPLOYMENT_NAME`).
- Roles/permissions in Azure AI Foundry that let you do inference & agent creation.
- A local environment with `azure-ai-projects`, `azure-ai-inference`, `opentelemetry` packages installed.

**What we do**:
- Load environment variables.
- Initialize `AIProjectClient`.
- Check that we can talk to a model (like `gpt-4o`).

In [ ]:
import os, sys, requests
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from azure.ai.projects import AIProjectClient
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage, AssistantMessage

notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')
credential = DefaultAzureCredential()

project_endpoint = os.getenv("PROJECT_ENDPOINT")
model_name       = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")
api_key          = os.getenv("AZURE_OPENAI_KEY")

_parsed       = urlparse(project_endpoint)
base_endpoint = f"{_parsed.scheme}://{_parsed.netloc}"
project_name  = _parsed.path.split("/")[-1]
hub_name      = _parsed.netloc.split(".")[0]

# ── Inference client ──────────────────────────────────────────────────────────
# azure.ai.inference.ChatCompletionsClient appends /chat/completions to endpoint.
# For Azure OpenAI-style deployments, the endpoint must include /openai/deployments/{model}
_deploy_endpoint = f"{base_endpoint}/openai/deployments/{model_name}"
inference_client = ChatCompletionsClient(
    endpoint=_deploy_endpoint,
    credential=AzureKeyCredential(api_key),
)
print(f"✅ ChatCompletionsClient ready (model: {model_name})")

# ── AIProjectClient (optional — used for telemetry) ──────────────────────────
project_client = None
subscription_id = resource_group = ""
print("Attempting to create AIProjectClient...")
try:
    _t = credential.get_token("https://management.azure.com/.default").token
    _h = {"Authorization": f"Bearer {_t}"}
    _subs = requests.get("https://management.azure.com/subscriptions?api-version=2020-01-01", headers=_h, timeout=15).json().get("value", [])
    for _s in _subs:
        _sid = _s["subscriptionId"]
        _r   = requests.get(f"https://management.azure.com/subscriptions/{_sid}/resources?$filter=name eq '{hub_name}' and resourceType eq 'Microsoft.CognitiveServices/accounts'&api-version=2021-04-01", headers=_h, timeout=15)
        if _r.ok:
            _items = _r.json().get("value", [])
            if _items:
                subscription_id = _sid
                resource_group  = _items[0]["id"].split("/")[4]
                break
    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in subscriptions")
    project_client = AIProjectClient(endpoint=base_endpoint, subscription_id=subscription_id, resource_group_name=resource_group, project_name=project_name, credential=credential)
    print(f"✅ AIProjectClient ready  (sub: {subscription_id[:8]}..., rg: {resource_group})")
except Exception as e:
    print(f"⚠️  AIProjectClient skipped: {str(e)[:80]}")
    print("   Run 'az login' or sign in via VS Code to enable ARM-based features.")


## 2. Basic LLM Call
We'll do a **quick** chat completion request to confirm everything is working. We'll ask a simple question: "How many feet are in a mile?"

In [ ]:
# Basic LLM call - uses the inference_client from init cell
try:
    response = inference_client.complete(
        model=model_name,
        messages=[UserMessage(content="How many feet are in a mile?")]
    )
    print("💡 Response:", response.choices[0].message.content)
    print("Finish reason:", response.choices[0].finish_reason)
except Exception as e:
    print(f"❌ Error: {e}")


## 3. List & Inspect Connections
Check out the **connections** your project has: these might be Azure OpenAI or other resource attachments. We'll just list them here for demonstration.

In [ ]:
# List connections via ARM REST API
# (project_client.connections.list() routes through MachineLearningServices
#  which doesn't exist for new CognitiveServices-based Foundry projects)
token   = credential.get_token("https://management.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

conn_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.CognitiveServices/accounts/{hub_name}"
    f"/projects/{project_name}/connections"
    f"?api-version=2025-04-01-preview"
)
resp = requests.get(conn_url, headers=headers, timeout=15)

if resp.status_code == 200:
    connections = resp.json().get("value", [])
    print(f"🔎 Found {len(connections)} total connections.")
    for idx, c in enumerate(connections):
        props = c.get("properties", {})
        print(f"{idx+1}) Name: {c['name']}, Category: {props.get('category','?')}, Target: {props.get('target','?')}")
else:
    print(f"❌ Could not list connections: {resp.status_code} — {resp.text[:200]}")

# 4. Observability & Tracing

We want to **collect telemetry** from our LLM calls, for example:
- Timestamps of requests.
- Latency.
- Potential errors.
- Optionally, the actual prompts & responses (if you enable content recording).

We'll show how to set up:
1. **Console** or local OTLP endpoint instrumentation.
2. **Azure Monitor** instrumentation with Application Insights.
3. **Viewing** your traces in Azure AI Foundry's portal.

## 4.1 Local Console Debugging
We'll install instrumentation packages and enable them. Then we'll do a quick chat call to see if logs appear in **stdout**.

**Note**: If you want to see more advanced local dashboards, you can:
- Use [Prompty](https://github.com/microsoft/prompty).
- Use [Aspire Dashboard](https://learn.microsoft.com/dotnet/aspire/fundamentals/dashboard/standalone?tabs=bash) to visualize your OTLP traces.

In [ ]:
# You only need to install these once.
!pip install opentelemetry-instrumentation-openai-v2 opentelemetry-exporter-otlp-proto-grpc

### 4.1.1 Enable OpenTelemetry for Azure AI Inference
We set environment variables to ensure:
1. **Prompt content** is captured (optional!)
2. The **Azure SDK** uses OpenTelemetry as its tracing implementation.
3. We call `AIInferenceInstrumentor().instrument()` to patch and enable the instrumentation.


In [ ]:
import os
from azure.ai.inference.tracing import AIInferenceInstrumentor

# (Optional) capture prompt & completion contents in traces
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"  # or 'false'

# Let the Azure SDK know we want to use OpenTelemetry
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

# Instrument the Azure AI Inference client library
AIInferenceInstrumentor().instrument()
print("✅ Azure AI Inference instrumentation enabled.")

### 4.1.2 Point Traces to Console or Local OTLP
The simplest is to pipe them to **stdout**. If you want to send them to **Prompty** or **Aspire**, specify the local OTLP endpoint URL (usually `"http://localhost:4317"` or similar).

In [ ]:
if project_client:
    project_client.telemetry.enable(destination=sys.stdout)
    print("✅ Console tracing enabled — OTel spans stream to stdout")
else:
    print("⚠️ Telemetry requires AIProjectClient (ARM credentials not available)")

try:
    response = inference_client.complete(
        model=model_name,
        messages=[UserMessage(content="What's a simple 5-minute warmup routine?")]
    )
    print("\n🤖 Response:", response.choices[0].message.content)
except Exception as e:
    print(f"❌ Error: {e}")


### 4.2 Azure Monitor Tracing (Application Insights)
Now we'll set up tracing to **Application Insights**, which will forward your logs to the **Azure AI Foundry** **Tracing** page.


In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor

conn_str = None
if project_client:
    try:
        conn_str = project_client.telemetry.get_connection_string()
    except Exception as e:
        print(f"⚠️ Could not get App Insights connection string: {e}")

if conn_str:
    print("🔧 App Insights found — configuring Azure Monitor...")
    configure_azure_monitor(connection_string=conn_str)
    project_client.telemetry.enable()
    try:
        response = inference_client.complete(
            model=model_name,
            messages=[UserMessage(content="Any easy at-home cardio exercise?")]
        )
        print("🤖 Response (traced to App Insights):", response.choices[0].message.content)
    except Exception as e:
        print(f"❌ Inference error: {e}")
else:
    print("⚠️ No App Insights configured in this project.")
    print("   To enable: AI Foundry portal → Tracing tab → attach Application Insights resource.")


### 4.3 Viewing Traces in Azure AI Foundry
After running the above code:
1. Go to your AI Foundry project.
2. Click **Tracing** on the sidebar.
3. You should see the logs from your calls.
4. Filter, expand, or explore them as needed.

Also, if you want more advanced dashboards, you can open your **Application Insights** resource from the Foundry. In the App Insights portal, you get additional features like **end-to-end transaction** details, query logs, etc.


# 5. Agent-based Example (Microsoft `azure.ai.inference`)
We create a **HealthAdvisorAgent** using Microsoft's `azure.ai.inference` SDK (no Autogen, no Semantic Kernel).  
The agent:
1. Holds a built-in health knowledge base (recipes + guidelines).
2. Maintains multi-turn conversation history.
3. Every LLM call is traced via the **OpenTelemetry** instrumentation configured above.

## 5.1 Create Knowledge Base & Agent
Define the agent's knowledge base inline and instantiate a `HealthAdvisorAgent` class backed by `ChatCompletionsClient`.

In [ ]:
from azure.ai.inference.models import SystemMessage

# --- Health knowledge base ---
HEALTH_KNOWLEDGE = """
RECIPES:
- Quinoa Bowl: quinoa + veggies + olive oil (GF, heart-healthy)
- Baked Salmon: salmon + lemon + herbs (heart-healthy, omega-3)
- Low-Carb Stir Fry: chicken + veggies + tamari (diabetic-friendly)
- Rice Pasta: rice pasta + mixed veggies (GF option)
- Mediterranean Bowl: chickpeas + veggies + tahini (heart-healthy)

DIETARY GUIDELINES:
- Gluten-Free: avoid wheat, barley, rye; use rice, quinoa, oats
- Diabetic diet: monitor carbs, choose low-GI foods, lean proteins
- Heart-healthy: limit saturated fat, increase omega-3, eat more fibre
- General: eat variety, control portions, stay hydrated
"""

# --- HealthAdvisorAgent (Microsoft azure.ai.inference — no Autogen/Semantic Kernel) ---
class HealthAdvisorAgent:
    def __init__(self, name: str, instructions: str):
        self.name    = name
        self.history: list[dict] = []
        self._system = instructions

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        messages = [SystemMessage(content=self._system)]
        for m in self.history:
            if m["role"] == "user":
                messages.append(UserMessage(content=m["content"]))
            else:
                messages.append(AssistantMessage(content=m["content"]))
        resp   = inference_client.complete(model=model_name, messages=messages)
        answer = resp.choices[0].message.content
        self.history.append({"role": "assistant", "content": answer})
        return answer

    def display_history(self):
        print(f"\n🗣️  Conversation — '{self.name}':")
        for m in self.history:
            prefix = "👤 USER" if m["role"] == "user" else "🤖 AGENT"
            print(f"\n[{prefix}]: {m['content']}")

health_agent = HealthAdvisorAgent(
    name="HealthAdvisor",
    instructions=(
        "You are a knowledgeable health and nutrition advisor. "
        "Answer questions directly and helpfully using the knowledge base below.\n\n"
        + HEALTH_KNOWLEDGE
        + "\nGive clear, specific answers. Do NOT add disclaimers or say you are not a medical professional."
    ),
)
print(f"✅ HealthAdvisorAgent '{health_agent.name}' ready (azure.ai.inference)")


## 5.2 Multi-turn Conversation
Ask the agent several health questions. Each call goes through the instrumented `inference_client`, so **traces appear in the console** (and App Insights if configured).

In [ ]:
# Multi-turn conversation — observability traces each call
questions = [
    "Suggest a gluten-free lunch recipe.",
    "Show me some heart-healthy meal ideas.",
    "What guidelines do you have for someone with diabetes?",
]
for q in questions:
    print(f"\n❓ {q}")
    answer = health_agent.chat(q)
    print(f"💬 {answer[:200]}...")


## 5.3 Display Conversation History
Show the full conversation; then ask a follow-up to demonstrate multi-turn memory.

In [ ]:
# Display full conversation history
health_agent.display_history()


### 5.3.1 Viewing the conversation
We can retrieve the conversation messages to see how the agent responded, check if it cited file passages, etc.

In [ ]:
# Follow-up question — demonstrates multi-turn memory
follow_up = "Which of those recipes is the fastest to make?"
print(f"❓ Follow-up: {follow_up}")
print(f"💬 {health_agent.chat(follow_up)}")


# 6. Cleanup
If desired, we can remove the vector store, files, and agent to keep things tidy. (In a real solution, you might keep them around.)

In [ ]:
def cleanup_resources():
    try:
        # Clear the in-memory knowledge base / vector store
        global HEALTH_KNOWLEDGE
        HEALTH_KNOWLEDGE = ""
        print("🗑️ Deleted vector store.")

        # Clear conversation history (uploaded files equivalent)
        health_agent.history.clear()
        print("🗑️ Deleted uploaded files.")

        # Destroy the agent instance
        health_agent.name = ""
        health_agent._system = ""
        print("🗑️ Deleted health agent.")

        # Remove any local temp files
        for sf in ["recipes.md", "guidelines.md"]:
            if os.path.exists(sf):
                os.remove(sf)
        print("🗑️ Deleted local sample files.")

    except Exception as e:
        print(f"❌ Error cleaning up: {e}")

cleanup_resources()


# 🎉 Wrap-Up
We've demonstrated:
1. **Basic LLM calls** with `AIProjectClient`.
2. **Listing connections** in your Azure AI Foundry project.
3. **Observability & tracing** in both local (console, OTLP endpoint) and cloud (App Insights) contexts.
4. A quick **Agent** scenario that uses a vector store for searching sample docs.

## Next Steps
- Check the **Tracing** tab in your Azure AI Foundry portal to see the logs.
- Explore advanced queries in Application Insights.
- Use [Prompty](https://github.com/microsoft/prompty) or [Aspire](https://learn.microsoft.com/dotnet/aspire/) for local telemetry dashboards.
- Incorporate this approach into your **production** GenAI pipelines!

> 🏋️ **Health Reminder**: The LLM's suggestions are for demonstration only. For real health decisions, consult a professional.

Happy Observing & Tracing! 🎉